In [11]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [12]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# BB84 Overview:
# Alice wants to share a secret key with Bob over a quantum channel.
# 1. Alice generates random bits and random bases (standard or diagonal).
# 2. Alice encodes each bit as a qubit using her chosen basis and sends it to Bob.
# 3. Bob independently chooses a random basis for each qubit and measures it.
# 4. Alice and Bob publicly compare their bases (NOT the bits).
# 5. They keep only the bits where their bases matched — this forms the sifted key.
# 6. They sacrifice a subset of the sifted key to check for eavesdropping.
#    If the error rate is below a threshold, the channel is considered secure.

# All random choices are generated by measuring a qubit in the |+> = 1/sqrt(2)(|0>+|1>) state,
# which gives a truly random 0 or 1 with equal probability.
# We do NOT use Python's random module.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# QUANTUM RANDOM NUMBER GENERATOR
# ─────────────────────────────────────────────────────────────────────────────
# We use quantum measurement to generate random bits.
# A Hadamard gate on |0> produces the superposition 1/sqrt(2)(|0>+|1>).
# Measuring this state collapses it to 0 or 1 with equal probability.

backend = BasicSimulator()

def quantum_random_bit():
    """Return a single random bit (0 or 1) via quantum measurement."""
    # Build a 1-qubit, 1-classical-bit circuit
    qc = QuantumCircuit(1, 1)
    # Apply Hadamard to create equal superposition |+>
    qc.h(0)
    # Measure the qubit into the classical bit
    qc.measure(0, 0)
    # Transpile and run with a single shot
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    # The only key in counts is the measured bit string
    return int(list(counts.keys())[0])


def quantum_random_bits(n):
    """Return a list of n random bits generated via quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]


# Quick sanity check — should print a list of 0s and 1s
sample = quantum_random_bits(10)
print("Sample random bits:", sample)

Sample random bits: [0, 0, 0, 1, 0, 0, 0, 0, 1, 1]


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# QUBIT ENCODING AND MEASUREMENT HELPERS
# ─────────────────────────────────────────────────────────────────────────────
# BB84 uses two bases:
#   Standard basis (basis=0):  |0> encodes bit 0,  |1> encodes bit 1
#   Diagonal basis (basis=1):  |+> encodes bit 0,  |-> encodes bit 1
#
# Encoding:
#   Standard: start in |0>; apply X gate if bit=1  → gives |0> or |1>
#   Diagonal: start in |0>; apply X if bit=1; then apply H → gives |+> or |->
#
# Measurement:
#   Standard: measure directly in the Z basis (distinguishes |0> from |1>)
#   Diagonal: apply H first to rotate back to Z basis, then measure
#             (H|+> = |0>, H|-> = |1>)

def encode_qubit(bit, basis):
    """
    Encode a classical bit into a qubit using the specified basis.

    Parameters
    ----------
    bit   : int  — 0 or 1, the value to encode
    basis : int  — 0 for standard, 1 for diagonal

    Returns
    -------
    QuantumCircuit — a 1-qubit circuit representing the encoded qubit
    """
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)          # Flip |0> to |1>
    if basis == 1:
        qc.h(0)          # Rotate to diagonal basis
    return qc


def measure_qubit(qubit_circuit, basis):
    """
    Measure a qubit circuit in the specified basis.

    Parameters
    ----------
    qubit_circuit : QuantumCircuit — the encoded qubit (1-qubit, no classical bits)
    basis         : int            — 0 for standard, 1 for diagonal

    Returns
    -------
    int — the measured bit (0 or 1)
    """
    # Copy the circuit and add a classical bit for measurement
    qc = qubit_circuit.copy()
    qc.add_register(__import__('qiskit').ClassicalRegister(1))
    if basis == 1:
        qc.h(0)          # Rotate back from diagonal basis before measuring
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# ALICE — KEY PREPARATION
# ─────────────────────────────────────────────────────────────────────────────
# Alice generates:
#   - A random bit string (her raw key)
#   - A random basis string (one basis per bit)
# She then encodes each bit as a qubit and "sends" it to Bob.

def alice_prepare(n):
    """
    Alice prepares n qubits for transmission.

    Parameters
    ----------
    n : int — number of qubits to prepare

    Returns
    -------
    bits   : list[int]           — Alice's random bit values
    bases  : list[int]           — Alice's random basis choices
    qubits : list[QuantumCircuit] — encoded qubits to send to Bob
    """
    bits  = quantum_random_bits(n)   # Random bit values
    bases = quantum_random_bits(n)   # Random basis choices (0=standard, 1=diagonal)
    qubits = [encode_qubit(b, basis) for b, basis in zip(bits, bases)]
    return bits, bases, qubits

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# BOB — MEASUREMENT
# ─────────────────────────────────────────────────────────────────────────────
# Bob receives the qubits from Alice.
# He independently chooses a random basis for each qubit and measures it.

def bob_measure(qubits):
    """
    Bob measures each received qubit in a randomly chosen basis.

    Parameters
    ----------
    qubits : list[QuantumCircuit] — qubits received from Alice

    Returns
    -------
    measured_bits : list[int] — Bob's measurement outcomes
    bases         : list[int] — Bob's random basis choices
    """
    bases = quantum_random_bits(len(qubits))   # Bob's random basis choices
    measured_bits = [measure_qubit(q, b) for q, b in zip(qubits, bases)]
    return measured_bits, bases

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# SIFTING — BASIS RECONCILIATION
# ─────────────────────────────────────────────────────────────────────────────
# Alice and Bob publicly compare their basis choices.
# They keep only the positions where both chose the same basis.
# The corresponding bits form the "sifted key".

def sift_key(alice_bits, alice_bases, bob_bits, bob_bases):
    """
    Perform basis sifting: retain bits where Alice and Bob used the same basis.

    Parameters
    ----------
    alice_bits  : list[int] — Alice's original bit values
    alice_bases : list[int] — Alice's basis choices
    bob_bits    : list[int] — Bob's measured bit values
    bob_bases   : list[int] — Bob's basis choices

    Returns
    -------
    alice_key : list[int] — Alice's sifted key
    bob_key   : list[int] — Bob's sifted key
    """
    alice_key = []
    bob_key   = []
    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:   # Bases match — keep this bit
            alice_key.append(alice_bits[i])
            bob_key.append(bob_bits[i])
    return alice_key, bob_key

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# ERROR CHECKING
# ─────────────────────────────────────────────────────────────────────────────
# Alice and Bob sacrifice a portion of their sifted key to estimate the
# quantum bit error rate (QBER).  A high QBER indicates eavesdropping.
# In the no-attacker scenario the QBER should be 0 (or very close to 0).

# Detection threshold: if QBER exceeds this fraction, abort the protocol.
DETECTION_THRESHOLD = 0.15   # 15 % error rate

def check_errors(alice_key, bob_key, sample_fraction=0.5):
    """
    Estimate the QBER by comparing a random sample of the sifted key.

    Parameters
    ----------
    alice_key       : list[int] — Alice's sifted key
    bob_key         : list[int] — Bob's sifted key
    sample_fraction : float     — fraction of the key to use for error checking

    Returns
    -------
    error_rate   : float     — proportion of mismatched bits in the sample
    attack_detected : bool   — True if error_rate exceeds DETECTION_THRESHOLD
    """
    n_sample = max(1, int(len(alice_key) * sample_fraction))
    # Use the first n_sample bits as the check sample
    errors = sum(a != b for a, b in zip(alice_key[:n_sample], bob_key[:n_sample]))
    error_rate = errors / n_sample
    attack_detected = error_rate > DETECTION_THRESHOLD
    return error_rate, attack_detected

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# PROTOCOL TABLE DISPLAY
# ─────────────────────────────────────────────────────────────────────────────
# Prints a formatted table matching the lecture slide layout:
#
#   Index   :  0   1   2  ...
#   A bit   :  0   1   0  ...
#   A basis :  s   d   s  ...
#   Qubit   :  0   -   0  ...
#   B basis :  s   s   d  ...
#   B bit   :  0   ?   ?  ...
#
# Positions where Alice and Bob's bases match are marked with  *  in the header.
# Qubit symbols: standard basis → 0 / 1 ; diagonal basis → + / -

def qubit_symbol(bit, basis):
    """Return the qubit state symbol for a given bit and basis."""
    if basis == 0:               # Standard basis: |0> or |1>
        return str(bit)
    else:                        # Diagonal basis: |+> or |->
        return '+' if bit == 0 else '-'


def print_bb84_table(alice_bits, alice_bases, bob_bits, bob_bases):
    """
    Print a BB84 protocol table to the console.

    Columns where Alice and Bob chose the same basis (sifted positions)
    are highlighted with an asterisk (*) below the index.
    Bob's bit is shown as '?' where bases did not match.

    Parameters
    ----------
    alice_bits  : list[int] — Alice's raw bit values
    alice_bases : list[int] — Alice's basis choices (0=standard, 1=diagonal)
    bob_bits    : list[int] — Bob's measured bit values
    bob_bases   : list[int] — Bob's basis choices (0=standard, 1=diagonal)
    """
    n = len(alice_bits)
    col = 3   # fixed column width

    # Determine which positions have matching bases
    match = [alice_bases[i] == bob_bases[i] for i in range(n)]

    # Build each row as a list of cell strings
    idx_row    = [str(i)                                          for i in range(n)]
    match_row  = ['*' if match[i] else ' '                        for i in range(n)]
    a_bit_row  = [str(alice_bits[i])                              for i in range(n)]
    a_bas_row  = ['s' if alice_bases[i] == 0 else 'd'             for i in range(n)]
    qubit_row  = [qubit_symbol(alice_bits[i], alice_bases[i])     for i in range(n)]
    b_bas_row  = ['s' if bob_bases[i] == 0 else 'd'               for i in range(n)]
    b_bit_row  = [str(bob_bits[i]) if match[i] else '?'           for i in range(n)]

    def fmt_row(label, cells):
        """Format a labelled row with fixed-width columns."""
        return f"{label:<10}" + "".join(c.center(col) for c in cells)

    # Separator line
    sep = '-' * (10 + col * n)

    print(sep)
    print(fmt_row('Index',   idx_row))
    print(fmt_row('Match',   match_row))   # * marks sifted positions
    print(sep)
    print(fmt_row('A bit',   a_bit_row))
    print(fmt_row('A basis', a_bas_row))   # s = standard, d = diagonal
    print(fmt_row('Qubit',   qubit_row))   # 0/1 for standard, +/- for diagonal
    print(sep)
    print(fmt_row('B basis', b_bas_row))
    print(fmt_row('B bit',   b_bit_row))   # ? where bases did not match
    print(sep)


# ─────────────────────────────────────────────────────────────────────────────
# FULL BB84 PROTOCOL — NO ATTACKER
# ─────────────────────────────────────────────────────────────────────────────
# Runs the complete BB84 protocol between Alice and Bob with no eavesdropper.

def run_bb84_plain(n_qubits=20):
    """
    Simulate the BB84 protocol without an attacker.

    Parameters
    ----------
    n_qubits : int — number of qubits Alice sends

    Returns
    -------
    final_key : list[int] — the shared secret key (remaining after error check)
    """
    print(f"=== BB84 Protocol (No Attacker) — {n_qubits} qubits ===")

    # ── ALICE ──────────────────────────────────────────────────────────────
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # ── QUANTUM CHANNEL ────────────────────────────────────────────────────
    # In the no-attacker scenario the qubits travel undisturbed.

    # ── BOB ────────────────────────────────────────────────────────────────
    bob_bits, bob_bases = bob_measure(qubits)

    # ── PROTOCOL TABLE ─────────────────────────────────────────────────────
    # Display the full per-qubit breakdown as a table (matches lecture slide).
    # Columns marked * are the sifted positions (bases matched).
    # B bit shows ? where bases differed — Bob's result is discarded there.
    print()
    print_bb84_table(alice_bits, alice_bases, bob_bits, bob_bases)

    # ── SIFTING (public channel) ────────────────────────────────────────────
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f"\n[Sifting] Alice sifted key : {alice_key}")
    print(f"[Sifting] Bob   sifted key : {bob_key}")

    # ── ERROR CHECKING ──────────────────────────────────────────────────────
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f"\n[Error Check] QBER = {error_rate:.2%}")

    if attack_detected:
        print("[Error Check] ⚠  Attack detected! Aborting protocol.")
        return []
    else:
        print("[Error Check] ✓  No attack detected. Channel is secure.")

    # ── FINAL KEY ───────────────────────────────────────────────────────────
    # Discard the bits used for error checking; the rest form the final key.
    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]   # Alice and Bob both keep the same suffix
    print(f"\n[Key] Final shared key ({len(final_key)} bits): {final_key}")
    return final_key


# Run the protocol
key = run_bb84_plain(n_qubits=20)

=== BB84 Protocol (No Attacker) — 20 qubits ===

----------------------------------------------------------------------
Index      0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19
Match      *           *  *  *  *        *     *  *  *           *    
----------------------------------------------------------------------
A bit      1  0  0  1  0  1  1  0  1  0  0  0  0  1  0  1  1  1  0  0 
A basis    d  d  d  d  s  s  s  s  d  d  d  s  d  d  s  d  s  s  d  s 
Qubit      -  +  +  -  0  1  1  0  -  +  +  0  +  -  0  -  1  1  +  0 
----------------------------------------------------------------------
B basis    d  s  s  s  s  s  s  s  s  s  d  d  d  d  s  s  d  d  d  d 
B bit      1  ?  ?  ?  0  1  1  0  ?  ?  0  ?  0  1  0  ?  ?  ?  0  ? 
----------------------------------------------------------------------

[Sifting] Alice sifted key : [1, 0, 1, 1, 0, 0, 0, 1, 0, 0]
[Sifting] Bob   sifted key : [1, 0, 1, 1, 0, 0, 0, 1, 0, 0]

[Error Check] QBER = 0.00%
[Error Check] ✓  No a

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# VERIFICATION
# ─────────────────────────────────────────────────────────────────────────────
# In the no-attacker scenario Alice and Bob should always agree on the sifted
# key bits.  We verify this by running the protocol multiple times and
# confirming the QBER is always 0.

print("Running 5 trials to verify zero error rate in the no-attacker scenario...\n")
for trial in range(1, 6):
    alice_bits, alice_bases, qubits = alice_prepare(30)
    bob_bits, bob_bases = bob_measure(qubits)
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    mismatches = sum(a != b for a, b in zip(alice_key, bob_key))
    print(f"  Trial {trial}: sifted key length = {len(alice_key)}, "
          f"mismatches = {mismatches}")

print("\nAll mismatches should be 0 — confirming correct protocol operation.")

Running 5 trials to verify zero error rate in the no-attacker scenario...

  Trial 1: sifted key length = 14, mismatches = 0
  Trial 2: sifted key length = 17, mismatches = 0
  Trial 3: sifted key length = 14, mismatches = 0
  Trial 4: sifted key length = 14, mismatches = 0
  Trial 5: sifted key length = 17, mismatches = 0

All mismatches should be 0 — confirming correct protocol operation.
